# Demo 3 — Regex, parsing, retrieval

Three ways to get something useful out of text, and they are **not**
alternatives. Each one answers a different question:

| | the question it answers |
|---|---|
| **regex** | is this text in the shape I expected? |
| **parsing** | is this data valid, and may it enter my program? |
| **retrieval** | which of my documents is this about? |

The mistake is using one where another belongs. This notebook shows each doing
its job, then each failing at somebody else's.

Runs offline. Nothing to install.

In [1]:
# Setup
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "pyproject.toml").exists():
    raise SystemExit(f"No course found above {Path.cwd()}. Open this inside your checkout.")
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"
print("ready")

ready


## 1. Regex — a shape, and nothing more

A regular expression asks one question: *does this text match this shape?*

Paste any of these into **[regexper.com](https://regexper.com/)** to see the same
thing as a railway diagram — it makes the assumptions visible.

In [2]:
import re

ORDER = re.compile(r"\border\s+(\d{4})\b", re.IGNORECASE)

messages = [
    "hey my order 4471 never showed up",
    "ORDER 4472 arrived damaged",
    "Order #4473 was charged twice",        # a '#' the pattern never allowed
    "my order, 4474, is late",              # a comma
    "order four thousand four hundred",     # words
]

for message in messages:
    found = ORDER.search(message)
    print(f"{'OK  ' if found else 'MISS'}  {found.group(1) if found else '-':>6}  {message}")

OK      4471  hey my order 4471 never showed up
OK      4472  ORDER 4472 arrived damaged
MISS       -  Order #4473 was charged twice
MISS       -  my order, 4474, is late
MISS       -  order four thousand four hundred


Three hits, two misses — and the two misses are the lesson.

`#4473` and `, 4474,` are perfectly ordinary human writing. The pattern did not
*fail*; it did exactly what it was told. **A regex encodes an assumption about
shape, and text written by people (or by a model) breaks assumptions.**

Now watch what makes it dangerous.

In [3]:
def urgency_from_prose(text: str) -> float:
    """Pull a confidence out of prose with a regex. This is the trap."""
    match = re.search(r"(\d+)\s*%\s*(?:sure|confident)", text, re.IGNORECASE)
    return int(match.group(1)) / 100 if match else 0.0


for reply in [
    "I am 90% sure this order was never delivered.",
    "I'm 90 percent confident this order was never delivered.",
    "Confidence: high. This order was never delivered.",
]:
    print(f"{urgency_from_prose(reply):.2f}   {reply}")

0.90   I am 90% sure this order was never delivered.
0.00   I'm 90 percent confident this order was never delivered.
0.00   Confidence: high. This order was never delivered.


Read the second and third rows again.

Both say the model is confident. Both return **`0.00`** — the same value as
*"I have no idea"*. Nothing raised. Nothing logged. A ticket that should have
been escalated sits in the automatic queue, and the only evidence is a number
that looks plausible.

> **This is the single most important thing on this page.** A regex that stops
> matching does not break loudly. It returns your default, silently, forever.

## 2. Parsing — a gate with a reason

Parsing asks a different question: *is this data valid, and may it enter my
program?* Every refusal has to say **which rule** stopped it.

In [4]:
from bootcamp_agent.schema import AnswerParseError, parse_research_answer

for raw in [
    '{"answer": "Chunking splits documents.", "citations": ["rag-basics"], '
    '"confidence": 0.85, "needs_human_review": false}',
    "I am 90% sure this is about chunking.",
    '{"answer": "x", "citations": [], "confidence": 7, "needs_human_review": false}',
]:
    try:
        parsed = parse_research_answer(raw)
        print(f"accepted  confidence={parsed.confidence}")
    except AnswerParseError as error:
        print(f"rejected  {error}")

accepted  confidence=0.85
rejected  Not valid JSON: Expecting value: line 1 column 1 (char 0)
rejected  'confidence' out of range [0, 1]: 7


Compare that with section 1.

The regex returned `0.0` and told you nothing. The parser returned an error that
**names the field and the rule**: `'confidence' out of range [0, 1]: 7`.

| | when the text is unexpected |
|---|---|
| regex | returns a default. You find out later, or never. |
| parser | refuses, and says which gate and which field. |

A regex is the right tool when you control the shape — a log line, a filename, a
date you wrote. It is the wrong tool for anything a model or a stranger wrote.

## 3. Retrieval — which document is this about?

Neither of the above can answer that. Retrieval scores your documents against a
query and hands back the ones that might support an answer.

In [5]:
from bootcamp_agent.documents import load_corpus
from bootcamp_agent.retrieval import retrieve

documents = load_corpus(CORPUS_DIR)
print(f"corpus: {len(documents)} documents\n")

for query in [
    "how does chunking work?",
    "how do I keep an assistant safe?",
    "how do I repot an orchid?",
]:
    hits = retrieve(query, documents, top_k=3)
    names = [scored.chunk.doc_id for scored in hits] or ["nothing"]
    print(f"{query:38} -> {names}")

corpus: 6 documents

how does chunking work?                -> ['evaluation-basics', 'prompt-injection', 'rag-basics']
how do I keep an assistant safe?       -> ['mcp-overview', 'mcp-overview', 'prompt-injection']
how do I repot an orchid?              -> ['nothing']


Three queries, three different *kinds* of answer — and the first one is not what
you would hope.

**"how does chunking work?"** returns three documents, and the top-ranked one is
`evaluation-basics`, not `rag-basics`. The lexical index is matching common words,
not meaning. It is not broken; it is a baseline, and this is what a baseline
looks like when you actually measure it instead of assuming.

**"how do I keep an assistant safe?"** straddles two documents, which is the
correct behaviour for a question that genuinely spans both.

**"how do I repot an orchid?"** returns **nothing**, and that is a *result*, not
an error. An empty retrieval is your program saying *"I have nothing that
supports an answer to this"* — and everything downstream can then refuse honestly
instead of inventing.

That first row is why session 6 builds the baseline and session 7 **measures**
it. A ranking tells you what is most similar. It does not tell you what is
right.

In [6]:
# What is inside a hit, and why the score is not a truth value.
for scored in retrieve("how does chunking work?", documents, top_k=2):
    chunk = scored.chunk
    print(f"{chunk.doc_id:20} position={chunk.position}  score={scored.score}")
    print(f"  {chunk.text.strip()[:110]}...")
    print()

evaluation-basics    position=0  score=2.525729
  # Evaluation Basics

"It seemed to work when I tried it" is not evidence. Evaluation replaces
anecdotes with a...

prompt-injection     position=2  score=2.525729
  No single defense is complete, but layers work. **Mark boundaries**: wrap
retrieved content in delimiters and ...



Look at the two scores: they are **identical**. The ranking between them is
arbitrary — decided by whatever order the documents happened to load in.

A score ranks; it does not verify. The top result is the *most similar* passage
by one crude measure, which is not the same as a *correct* one. Session 7 is
about measuring that gap instead of hoping it is small.

## 4. The three together, on one message

A real pipeline uses all three, each for its own job.

In [7]:
def triage(message: str) -> dict:
    """regex finds the shape · retrieval finds the topic · parsing guards the exit."""
    order = ORDER.search(message)
    hits = retrieve(message, documents, top_k=2)
    return {
        "order_id": int(order.group(1)) if order else None,
        "about": sorted({scored.chunk.doc_id for scored in hits}) or None,
        "needs_human": order is None,
    }


for message in [
    "order 4471 never arrived, and I am worried about prompt injection in your bot",
    "do you ship to Portugal?",
]:
    print(f"{message}\n  -> {triage(message)}\n")

order 4471 never arrived, and I am worried about prompt injection in your bot
  -> {'order_id': 4471, 'about': ['prompt-injection', 'structured-outputs'], 'needs_human': False}

do you ship to Portugal?
  -> {'order_id': None, 'about': None, 'needs_human': True}



The second message has no order number **and** retrieves nothing, so it is
flagged for a human. Two independent signals agreeing is a much better reason to
escalate than either alone.

## What to take away

- **regex** answers *"is this the shape I expected?"* — and returns a default
  when it is not. Use it on text you control.
- **parsing** answers *"may this enter my program?"* — and every refusal names
  the rule. Use it on anything a model or a stranger wrote.
- **retrieval** answers *"which of my documents is this about?"* — and *nothing*
  is a legitimate answer.
- The dangerous failure is the silent one. A regex that stops matching keeps
  returning `0.0`, and nothing anywhere tells you.

Open **[regexper.com](https://regexper.com/)** and paste
`order\s+(\d{4})` into it. Seeing the shape as a diagram is the fastest way
to notice what it refuses to match.